# Smart City Traffic Management and Route Optimization
Experiments for the capstone project: road network modelling, Dijkstra, Bellman-Ford, BFS/DFS, Prim/Kruskal and a comparative analysis.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from network_model import RoadNetwork
from route_planner import dijkstra
from traffic_analysis import bellman_ford, apply_congestion, compare_routes
from connectivity_module import bfs, dfs
from mst_planner import prim, kruskal
from benchmark import run_benchmarks
from viz import draw_network, layout

## 1. Road network model
Intersections are vertices, roads are edges, and weights are travel times in minutes.

In [2]:
net = RoadNetwork.sample_city()
pos = layout(net)
print(net.num_nodes, 'intersections,', net.num_edges, 'roads')
print(net.adjacency_list_text())

12 intersections, 18 roads
Airport -> (Highway Jn, 8), (Tech Park, 12)
Central Station -> (City Hall, 4), (Market, 6), (Stadium, 9), (University, 5)
City Hall -> (Central Station, 4), (Hospital, 8), (Market, 3)
Harbor -> (Mall, 9), (Old Town, 7)
Highway Jn -> (Airport, 8), (Stadium, 10), (Tech Park, 6)
Hospital -> (City Hall, 8), (Mall, 6), (Old Town, 10), (University, 11)
Mall -> (Harbor, 9), (Hospital, 6), (Stadium, 14)
Market -> (Central Station, 6), (City Hall, 3), (Old Town, 5)
Old Town -> (Harbor, 7), (Hospital, 10), (Market, 5)
Stadium -> (Central Station, 9), (Highway Jn, 10), (Mall, 14)
Tech Park -> (Airport, 12), (Highway Jn, 6), (University, 7)
University -> (Central Station, 5), (Hospital, 11), (Tech Park, 7)


In [3]:
net.adjacency_matrix()

,Airport,Central Station,City Hall,Harbor,Highway Jn,Hospital,Mall,Market,Old Town,Stadium,Tech Park,University
Airport,0,∞,∞,∞,8,∞,∞,∞,∞,∞,12,∞
Central Station,∞,0,4,∞,∞,∞,∞,6,∞,9,∞,5
City Hall,∞,4,0,∞,∞,8,∞,3,∞,∞,∞,∞
Harbor,∞,∞,∞,0,∞,∞,9,∞,7,∞,∞,∞
Highway Jn,8,∞,∞,∞,0,∞,∞,∞,∞,10,6,∞
Hospital,∞,∞,8,∞,∞,0,6,∞,10,∞,∞,11
Mall,∞,∞,∞,9,∞,6,0,∞,∞,14,∞,∞
Market,∞,6,3,∞,∞,∞,∞,0,5,∞,∞,∞
Old Town,∞,∞,∞,7,∞,10,∞,5,0,∞,∞,∞
Stadium,∞,9,∞,∞,10,∞,14,∞,∞,0,∞,∞


In [4]:
draw_network(net, pos, title='Sample city road network', figsize=(9,5.5)); plt.show()

## 2. Route planning with Dijkstra

In [5]:
res = dijkstra(net, 'Airport', 'Harbor')
print('Route:', ' -> '.join(res.path), '| cost:', res.cost)
pd.DataFrame(res.steps)

Route: Airport -> Highway Jn -> Stadium -> Mall -> Harbor | cost: 41.0


,Step,Settled node,Relax to,Road cost,Old distance,New distance
0,1,Airport,Tech Park,12.0,inf,12.0
1,2,Airport,Highway Jn,8.0,inf,8.0
2,3,Highway Jn,Stadium,10.0,inf,18.0
3,4,Tech Park,University,7.0,inf,19.0
4,5,Stadium,Central Station,9.0,inf,27.0
5,6,Stadium,Mall,14.0,inf,32.0
6,7,University,Central Station,5.0,27.0,24.0
7,8,University,Hospital,11.0,inf,30.0
8,9,Central Station,City Hall,4.0,inf,28.0
9,10,Central Station,Market,6.0,inf,30.0


In [6]:
draw_network(net, pos, path=res.path, title='Shortest route Airport -> Harbor'); plt.show()

## 3. Traffic analysis with Bellman-Ford
We triple the travel time on the Stadium-Mall road and recompute the route.

In [7]:
congested = apply_congestion(net, [('Mall','Stadium')], factor=3)
after = bellman_ford(congested, 'Airport', 'Harbor')
print('Before:', ' -> '.join(res.path), res.cost)
print('After :', ' -> '.join(after.path), after.cost)
compare_routes(res, after)

Before: Airport -> Highway Jn -> Stadium -> Mall -> Harbor 41.0
After : Airport -> Tech Park -> University -> Central Station -> Market -> Old Town -> Harbor 42.0


{'route_changed': True,
 'reachable_after': True,
 'original_cost': 41.0,
 'updated_cost': 42.0,
 'delta': 1.0,
 'pct': 2.4390243902439024}

In [8]:
fig, axs = plt.subplots(1, 2, figsize=(14,5))
for ax, (n_, p_, t_) in zip(axs, [(net, res.path, 'Original'), (congested, after.path, 'After congestion')]):
    import networkx as nx
    g = n_.to_networkx(); hl = set(zip(p_, p_[1:])) | set(zip(p_[1:], p_))
    nx.draw(g, pos, ax=ax, with_labels=True, node_size=500, font_size=6, node_color='#4C78A8', font_color='white',
            edge_color=['#E45756' if e in hl else '#B8B8B8' for e in g.edges], width=[3.5 if e in hl else 1 for e in g.edges])
    ax.set_title(t_)
plt.show()

## 4. Connectivity: BFS and DFS

In [9]:
b, d = bfs(net, 'Airport'), dfs(net, 'Airport')
print('BFS:', b['order'])
print('DFS:', d['order'])
print('Reachable:', len(b['reachable']), 'of', net.num_nodes)

BFS: ['Airport', 'Highway Jn', 'Tech Park', 'Stadium', 'University', 'Central Station', 'Mall', 'Hospital', 'City Hall', 'Market', 'Harbor', 'Old Town']
DFS: ['Airport', 'Highway Jn', 'Stadium', 'Central Station', 'City Hall', 'Hospital', 'Mall', 'Harbor', 'Old Town', 'Market', 'University', 'Tech Park']
Reachable: 12 of 12


## 5. Infrastructure planning: Prim and Kruskal

In [10]:
p, k = prim(net, 'Airport'), kruskal(net)
print('Prim total   :', p['total'])
print('Kruskal total:', k['total'])
pd.DataFrame(k['edges'], columns=['From','To','Cost'])

Prim total   : 68.0
Kruskal total: 68.0


,From,To,Cost
0,City Hall,Market,3.0
1,Central Station,City Hall,4.0
2,Central Station,University,5.0
3,Market,Old Town,5.0
4,Highway Jn,Tech Park,6.0
5,Hospital,Mall,6.0
6,Harbor,Old Town,7.0
7,Tech Park,University,7.0
8,Airport,Highway Jn,8.0
9,City Hall,Hospital,8.0


In [11]:
draw_network(net, pos, edges=p['edges'], color='#54A24B', title=f"MST (total cost {p['total']:g})"); plt.show()

## 6. Comparative analysis
Random connected networks with 10, 25, 50 and 100 intersections (about 2n roads); times are the mean of 50 runs.

In [12]:
bench = run_benchmarks(reps=50)
for name, df in bench.items():
    print(name); display(df.round(4))

BFS vs DFS


,Intersections,Roads,BFS time (ms),DFS time (ms),BFS memory (KB),DFS memory (KB),BFS visited,DFS visited
0,10,19,0.0057,0.0058,1.8281,1.5938,10,10
1,25,49,0.0122,0.0369,4.5000,5.2188,25,25
2,50,99,0.0313,0.0268,5.3906,5.3750,50,50
3,100,199,0.0508,0.0587,15.5469,19.3125,100,100


Dijkstra vs Bellman-Ford


,Intersections,Roads,Dijkstra time (ms),Bellman-Ford time (ms),Dijkstra memory (KB),Bellman-Ford memory (KB),Route cost,Route hops,Costs agree
0,10,19,0.0063,0.0215,1.3828,0.9219,7.0,1,True
1,25,49,0.0212,0.0495,4.3359,2.8203,22.0,3,True
2,50,99,0.0407,0.0979,6.2500,5.3672,42.0,5,True
3,100,199,0.0901,0.1602,17.4609,11.0703,34.0,4,True


Prim vs Kruskal


,Intersections,Roads,Prim time (ms),Kruskal time (ms),Prim memory (KB),Kruskal memory (KB),Prim MST cost,Kruskal MST cost
0,10,19,0.0282,0.0256,1.5938,1.5547,41.0,41.0
1,25,49,0.0662,0.0639,5.5625,5.2578,149.0,149.0
2,50,99,0.1309,0.1263,9.0391,10.6016,280.0,280.0
3,100,199,0.2739,0.2602,22.6797,21.6016,568.0,568.0


In [13]:
fig, axs = plt.subplots(1, 3, figsize=(15,4))
for ax, (name, df) in zip(axs, bench.items()):
    cols = [c for c in df.columns if c.endswith('time (ms)')]
    for c in cols: ax.plot(df['Intersections'], df[c], marker='o', label=c.replace(' time (ms)',''))
    ax.set_title(name); ax.set_xlabel('Intersections'); ax.set_ylabel('ms'); ax.legend()
plt.tight_layout(); plt.show()

## 7. Observations
- Dijkstra and Bellman-Ford always return the same cost; Dijkstra is faster because it avoids repeated passes over every edge.
- BFS and DFS have the same O(V+E) behaviour, so their times are nearly identical.
- Prim and Kruskal produce the same total MST cost; their speeds are very close on sparse road networks.
- Congesting one road can change the best route (a diversion) or only increase its cost.